In [1]:
!pip install s3fs pyarrow pandas duckdb -q

In [2]:
import s3fs
import pandas as pd

# Public access – no AWS credentials needed
fs = s3fs.S3FileSystem(anon=True)

# Base path of the dataset
base_path = "oedi-data-lake/pv-rooftop-pr/developable-planes"

# List all county partitions
partitions = fs.ls(base_path)

# Extract clean municipality names
municipalities = []
for p in partitions:
    if "county=" in p:
        name = p.split("county=")[-1]
        municipalities.append(name)

municipalities = sorted(municipalities)

print(f"Total municipalities found: {len(municipalities)}\n")
print(municipalities)

Total municipalities found: 78

['Adjuntas', 'Aguada', 'Aguadilla', 'Aguas Buenas', 'Aibonito', 'Arecibo', 'Arroyo', 'Añasco', 'Barceloneta', 'Barranquitas', 'Bayamón', 'Cabo Rojo', 'Caguas', 'Camuy', 'Canóvanas', 'Carolina', 'Cataño', 'Cayey', 'Ceiba', 'Ciales', 'Cidra', 'Coamo', 'Comerío', 'Corozal', 'Culebra', 'Dorado', 'Fajardo', 'Florida', 'Guayama', 'Guayanilla', 'Guaynabo', 'Gurabo', 'Guánica', 'Hatillo', 'Hormigueros', 'Humacao', 'Isabela', 'Jayuya', 'Juana Díaz', 'Juncos', 'Lajas', 'Lares', 'Las Marías', 'Las Piedras', 'Loíza', 'Luquillo', 'Manatí', 'Maricao', 'Maunabo', 'Mayagüez', 'Moca', 'Morovis', 'Naguabo', 'Naranjito', 'Orocovis', 'Patillas', 'Peñuelas', 'Ponce', 'Quebradillas', 'Rincón', 'Río Grande', 'Sabana Grande', 'Salinas', 'San Germán', 'San Juan', 'San Lorenzo', 'San Sebastián', 'Santa Isabel', 'Toa Alta', 'Toa Baja', 'Trujillo Alto', 'Utuado', 'Vega Alta', 'Vega Baja', 'Vieques', 'Villalba', 'Yabucoa', 'Yauco']


In [3]:
import duckdb

# Connect to DuckDB (works great with Parquet on S3)
con = duckdb.connect()

# Install and load httpfs extension for S3 access
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")
con.execute("SET s3_access_key_id='';")      # anonymous
con.execute("SET s3_secret_access_key='';")

# Query to get summary per municipality
query = f"""
SELECT 
    county AS municipality,
    COUNT(*) AS num_planes,
    COUNT(DISTINCT bldg_fid) AS num_buildings,
    ROUND(SUM(flat_m2), 1) AS total_flat_m2,
    ROUND(SUM(slope_m2), 1) AS total_slope_m2,
    ROUND(AVG(flat_m2), 2) AS avg_plane_flat_m2,
    ROUND(AVG(slope_m2), 2) AS avg_plane_slope_m2,
    ROUND(SUM(annual_kwh), 0) AS total_annual_kwh
FROM read_parquet('s3://oedi-data-lake/pv-rooftop-pr/developable-planes/county=*/**/*.parquet', 
                   hive_partitioning=1)
GROUP BY county
ORDER BY county
"""

df_summary = con.execute(query).df()

# Convert m² to ft² (optional)
df_summary["total_flat_ft2"] = (df_summary["total_flat_m2"] * 10.7639).round(0)
df_summary["total_slope_ft2"] = (df_summary["total_slope_m2"] * 10.7639).round(0)

print(df_summary.head(15))
print(f"\nTotal municipalities: {len(df_summary)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    municipality  num_planes  num_buildings  total_flat_m2  total_slope_m2  \
0       Adjuntas       33602           6327       511115.3        619224.9   
1         Aguada      107466          18089      1680211.6       2069484.4   
2      Aguadilla      147710          23698      2810391.7       3496288.0   
3   Aguas Buenas       63179          10858       926081.9       1147381.3   
4       Aibonito       77392          12629      1297154.5       1634089.7   
5        Arecibo      246439          43059      4766720.4       5781156.9   
6         Arroyo       49418           9054       856387.7       1013210.7   
7         Añasco       74529          13720      1202035.3       1466350.5   
8    Barceloneta       53104           9453      1145000.5       1417945.7   
9   Barranquitas       74731          12880      1182480.2       1440494.8   
10       Bayamón      416563          64260      9368398.2      11543889.1   
11     Cabo Rojo      150694          27209      2396488.6      

In [4]:
# Save to CSV / Excel
df_summary.to_csv("PVRDB_PR_Municipalities_Summary.csv", index=False)
df_summary.to_excel("PVRDB_PR_Municipalities_Summary.xlsx", index=False)

print("Files saved:")
print("- PVRDB_PR_Municipalities_Summary.csv")
print("- PVRDB_PR_Municipalities_Summary.xlsx")

Files saved:
- PVRDB_PR_Municipalities_Summary.csv
- PVRDB_PR_Municipalities_Summary.xlsx


In [5]:
def load_municipality(municipio_name: str):
    """
    Load all developable planes for a specific municipality.
    Example: load_municipality("San Juan")
    """
    path = f"s3://oedi-data-lake/pv-rooftop-pr/developable-planes/county={municipio_name}/**/*.parquet"
    
    df = con.execute(f"""
        SELECT *
        FROM read_parquet('{path}', hive_partitioning=1)
    """).df()
    
    print(f"Loaded {len(df):,} planes for {municipio_name}")
    return df

# Example usage:
# df_sj = load_municipality("San Juan")
# df_sj.head()

In [6]:
# Show the data dictionary columns
sample = con.execute("""
    SELECT * 
    FROM read_parquet('s3://oedi-data-lake/pv-rooftop-pr/developable-planes/county=San Juan/**/*.parquet', 
                      hive_partitioning=1)
    LIMIT 5
""").df()

print("Available columns:")
print(sample.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Available columns:
['devp_gid', 'bldg_plane_id', 'bldg_fid', 'bg_geoid', 'azimuth', 'tilt', 'flat_m2', 'slope_degrees', 'slope_m2', 'pitchmultiplier', 'pct_shading', 'area_derate_factor', 'kw', 'dc_ac_ratio', 'array_type', 'losses', 'module_type', 'inverter_efficiency', 'annual_kwh', 'the_geom_4326', 'the_geom_32620', 'county']


In [7]:
suitable = df_summary[df_summary["total_slope_m2"] > 50]  # example threshold

In [8]:
# ==============================================
# CONFIGURE YOUR MINIMUM ROOF SPACE (in ft²)
# ==============================================

MIN_AREA_FT2 = 130          # Minimum suitable roof area in square feet
# Common references:
# 130 ft² ≈ 12 m² (≈ 4-5 panels)
# 215 ft² ≈ 20 m² (≈ 7-8 panels)
# 320 ft² ≈ 30 m² (≈ 10-12 panels)
# 430 ft² ≈ 40 m² (comfortable residential system)

USE_SLOPE_AREA = True       # True = use actual sloping roof area
                            # False = use flat (bird's-eye) area

In [9]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")
con.execute("SET s3_access_key_id='';")
con.execute("SET s3_secret_access_key='';")

# Conversion factor
M2_TO_FT2 = 10.7639

query = f"""
SELECT 
    county AS municipality,
    bldg_fid,
    COUNT(*) AS num_planes,
    
    -- Original metric
    ROUND(SUM(flat_m2), 2) AS total_flat_m2,
    ROUND(SUM(slope_m2), 2) AS total_slope_m2,
    
    -- Converted to square feet
    ROUND(SUM(flat_m2) * {M2_TO_FT2}, 1) AS total_flat_ft2,
    ROUND(SUM(slope_m2) * {M2_TO_FT2}, 1) AS total_slope_ft2,
    
    ROUND(SUM(annual_kwh), 0) AS total_annual_kwh,
    
    -- Status based on square feet
    CASE 
        WHEN SUM({'slope_m2' if USE_SLOPE_AREA else 'flat_m2'}) * {M2_TO_FT2} >= {MIN_AREA_FT2} 
        THEN 'Suitable' 
        ELSE 'Insufficient' 
    END AS roof_status
FROM read_parquet(
    's3://oedi-data-lake/pv-rooftop-pr/developable-planes/county=*/**/*.parquet',
    hive_partitioning = 1
)
GROUP BY county, bldg_fid
"""

df_buildings = con.execute(query).df()

print(f"Total buildings processed: {len(df_buildings):,}")
print(f"Minimum area threshold: {MIN_AREA_FT2} ft²")
df_buildings.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total buildings processed: 1,343,874
Minimum area threshold: 130 ft²


,municipality,bldg_fid,num_planes,total_flat_m2,total_slope_m2,total_flat_ft2,total_slope_ft2,total_annual_kwh,roof_status
0,Aguada,383220,5,84.60,98.47,910.6,1059.9,17175.0,Suitable
1,Aguada,383169,4,77.48,82.68,834.0,890.0,16020.0,Suitable
2,Aguada,436652,4,22.56,29.14,242.9,313.6,5210.0,Suitable
3,Aguada,436122,4,65.06,69.51,700.3,748.2,11846.0,Suitable
4,Aguada,321844,3,84.23,85.02,906.6,915.1,13337.0,Suitable


In [10]:
summary = (
    df_buildings
    .groupby(["municipality", "roof_status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

summary["Total_Buildings"] = summary["Suitable"] + summary["Insufficient"]
summary["% Suitable"] = (summary["Suitable"] / summary["Total_Buildings"] * 100).round(1)
summary["% Insufficient"] = (summary["Insufficient"] / summary["Total_Buildings"] * 100).round(1)

# Sort by highest percentage of insufficient roofs
summary = summary.sort_values("% Insufficient", ascending=False)

print("=== Buildings with Insufficient Roof Space by Municipality ===\n")
display(summary)

=== Buildings with Insufficient Roof Space by Municipality ===



roof_status,municipality,Insufficient,Suitable,Total_Buildings,% Suitable,% Insufficient
41,Lares,1142,14360,15502,92.6,7.4
47,Maricao,246,3135,3381,92.7,7.3
42,Las Marías,314,4011,4325,92.7,7.3
71,Utuado,1128,14857,15985,92.9,7.1
56,Peñuelas,660,8781,9441,93.0,7.0
...,...,...,...,...,...,...
69,Toa Baja,457,26018,26475,98.3,1.7
16,Cataño,98,6894,6992,98.6,1.4
64,San Juan,1022,93706,94728,98.9,1.1
12,Caguas,146,39692,39838,99.6,0.4


In [11]:
insufficient = df_buildings[df_buildings["roof_status"] == "Insufficient"].copy()

print(f"Buildings with insufficient roof space (< {MIN_AREA_FT2} ft²): {len(insufficient):,}")
print(f"Percentage of all buildings: {len(insufficient)/len(df_buildings)*100:.1f}%\n")

# Show relevant columns in ft²
cols_to_show = ["municipality", "bldg_fid", "num_planes", 
                "total_flat_ft2", "total_slope_ft2", "total_annual_kwh", "roof_status"]

display(insufficient[cols_to_show].head(10))

Buildings with insufficient roof space (< 130 ft²): 52,466
Percentage of all buildings: 3.9%



,municipality,bldg_fid,num_planes,total_flat_ft2,total_slope_ft2,total_annual_kwh,roof_status
28,Aguada,382357,1,28.4,29.4,602.0,Insufficient
36,Aguada,258975,1,18.8,23.9,343.0,Insufficient
144,Aguada,406990,2,43.9,51.5,963.0,Insufficient
163,Aguada,402508,1,89.4,103.5,2384.0,Insufficient
202,Aguada,432036,3,102.4,109.4,2010.0,Insufficient
233,Aguada,1135214,1,32.7,34.0,465.0,Insufficient
280,Aguas Buenas,1382759,1,104.2,124.3,2300.0,Insufficient
316,Aguas Buenas,1325379,2,96.2,107.8,1711.0,Insufficient
317,Aguas Buenas,1263638,2,52.6,112.1,1546.0,Insufficient
345,Aguas Buenas,224146,2,72.4,106.7,2045.0,Insufficient


In [12]:
# 1. Summary by municipality
summary.to_csv("PR_Roof_Space_Summary_by_Municipality_ft2.csv", index=False)
summary.to_excel("PR_Roof_Space_Summary_by_Municipality_ft2.xlsx", index=False)

# 2. Buildings with insufficient roof space
insufficient[cols_to_show].to_csv("PR_Buildings_Insufficient_Roof_Space_ft2.csv", index=False)

# 3. Suitable buildings
suitable = df_buildings[df_buildings["roof_status"] == "Suitable"]
suitable[cols_to_show].to_csv("PR_Buildings_Suitable_Roof_Space_ft2.csv", index=False)

print("Files exported (all values in square feet):")
print("• PR_Roof_Space_Summary_by_Municipality_ft2.csv / .xlsx")
print("• PR_Buildings_Insufficient_Roof_Space_ft2.csv")
print("• PR_Buildings_Suitable_Roof_Space_ft2.csv")

Files exported (all values in square feet):
• PR_Roof_Space_Summary_by_Municipality_ft2.csv / .xlsx
• PR_Buildings_Insufficient_Roof_Space_ft2.csv
• PR_Buildings_Suitable_Roof_Space_ft2.csv


In [13]:
print("=== Quick Stats (Square Feet) ===")
print(f"Minimum area used: {MIN_AREA_FT2} ft²")
print(f"Total buildings analyzed: {len(df_buildings):,}")
print(f"Suitable buildings: {(df_buildings['roof_status']=='Suitable').sum():,}")
print(f"Insufficient buildings: {(df_buildings['roof_status']=='Insufficient').sum():,}")
print(f"% of houses with insufficient roof space: {(df_buildings['roof_status']=='Insufficient').mean()*100:.1f}%")

=== Quick Stats (Square Feet) ===
Minimum area used: 130 ft²
Total buildings analyzed: 1,343,874
Suitable buildings: 1,291,408
Insufficient buildings: 52,466
% of houses with insufficient roof space: 3.9%


In [14]:
!pip install geopandas matplotlib mapclassify -q

In [15]:
import os

print("Files in current directory:")
for f in sorted(os.listdir()):
    if f.lower().endswith(('.geojson', '.json', '.shp', '.gpkg', '.zip')):
        print(" →", f)

Files in current directory:
 → Tableau_PR_Municipios_Roof.geojson
 → us-states.json


In [16]:
import geopandas as gpd

url = "https://raw.githubusercontent.com/commonwealth-of-puerto-rico/crime-spotter/master/public/data/municipalities.geojson"

gdf = gpd.read_file(url)

print("gdf loaded successfully!")
print("Number of municipalities:", len(gdf))
print("\nColumns:", gdf.columns.tolist())
print("\nFirst 5 rows:")
display(gdf.head())

gdf loaded successfully!
Number of municipalities: 78

Columns: ['STATE', 'COUNTY', 'NAME', 'geometry']

First 5 rows:


,STATE,COUNTY,NAME,geometry
0,72,071,Isabela,"POLYGON ((-67.10328 18.51343, -67.10339 18.514..."
1,72,005,Aguadilla,"POLYGON ((-67.10328 18.51343, -67.10281 18.512..."
2,72,013,Arecibo,"POLYGON ((-66.58678 18.48495, -66.58683 18.484..."
3,72,065,Hatillo,"POLYGON ((-66.76483 18.48407, -66.76598 18.481..."
4,72,115,Quebradillas,"POLYGON ((-66.90143 18.48455, -66.90138 18.483..."


In [17]:
import unicodedata

def clean_muni_name(name):
    if pd.isna(name):
        return name
    # Remove accents
    name = unicodedata.normalize('NFKD', str(name)).encode('ASCII', 'ignore').decode('utf-8')
    # Common replacements
    name = name.replace(" Municipio", "").replace("Municipio", "")
    name = name.strip().title()
    return name

# Apply to both datasets
gdf["municipality"] = gdf["NAME"].apply(clean_muni_name)
summary["municipality"] = summary["municipality"].apply(clean_muni_name)

# Check again
print("After better cleaning:")
print("Missing in map:", sorted(set(summary["municipality"]) - set(gdf["municipality"])))

After better cleaning:
Missing in map: []


In [18]:
name_fix = {
    # "name_in_data": "name_in_map"
    "Anasco": "Anasco",
    "Rincon": "Rincon",
    "Guanica": "Guanica",
    "San Sebastian": "San Sebastian",
    "Loiza": "Loiza",
    "Manati": "Manati",
    "Mayaguez": "Mayaguez",
    "Bayamon": "Bayamon",
    "Catano": "Catano",
    "Juana Diaz": "Juana Diaz",
    "Penuelas": "Penuelas",
    # Add more if needed after running the diagnosis
}

# Apply the fix
summary["municipality"] = summary["municipality"].replace(name_fix)
gdf["municipality"] = gdf["municipality"].replace(name_fix)  # just in case

In [19]:
gdf_map = gdf.merge(summary, on="municipality", how="left")

print("Municipalities with data after fix:")
print(gdf_map["% Insufficient"].notna().sum(), "out of", len(gdf_map))

# Show which ones are still missing
still_missing = gdf_map[gdf_map["% Insufficient"].isna()]["municipality"].tolist()
print("\nStill missing:", still_missing)

Municipalities with data after fix:
78 out of 78

Still missing: []


In [20]:
import folium
from folium import Choropleth, GeoJson, GeoJsonTooltip, LayerControl

# Base map
m = folium.Map(
    location=[18.22, -66.48],
    zoom_start=9,
    tiles="CartoDB positron"
)

# Choropleth with Green → Yellow → Red
Choropleth(
    geo_data=gdf_map.__geo_interface__,
    data=gdf_map,
    columns=["municipality", "% Insufficient"],
    key_on="feature.properties.municipality",
    
    # Green (low) → Yellow → Red (high)
    fill_color="RdYlGn_r",          # ← This is the key change
    fill_opacity=0.78,
    line_opacity=0.6,
    line_color="#333333",
    line_weight=0.7,
    
    legend_name="% Buildings with Insufficient Roof Space",
    name="Insufficient Roof Space",
    
    bins=7
).add_to(m)

# Tooltips
GeoJson(
    gdf_map,
    style_function=lambda x: {
        "fillColor": "transparent",
        "color": "transparent",
        "weight": 0
    },
    tooltip=GeoJsonTooltip(
        fields=["municipality", "% Insufficient", "Insufficient", "Total_Buildings"],
        aliases=["Municipio:", "% Insufficient:", "Buildings Insufficient:", "Total Buildings:"],
        style=("background-color: white; color: #333; font-family: Arial; font-size: 12px; padding: 8px;")
    )
).add_to(m)

LayerControl().add_to(m)

# Save and show
m.save("PR_Roof_Space_Map_By_Percent.html")
print("Map saved as → PR_Roof_Space_Map_By_Percent.html")
m

Map saved as → PR_Roof_Space_Map_By_Percent.html


In [21]:
import folium
from folium import Choropleth, GeoJson, GeoJsonTooltip, LayerControl

# Base map
m = folium.Map(
    location=[18.22, -66.48],
    zoom_start=9,
    tiles="CartoDB positron"
)

# Choropleth by QUANTITY (absolute numbers)
Choropleth(
    geo_data=gdf_map.__geo_interface__,
    data=gdf_map,
    columns=["municipality", "Insufficient"],          # ← Using quantity
    key_on="feature.properties.municipality",
    
    # Green → Yellow → Red
    fill_color="RdYlGn_r",
    fill_opacity=0.78,
    line_opacity=0.6,
    line_color="#333333",
    line_weight=0.7,
    
    legend_name="Number of Buildings with Insufficient Roof Space",
    name="Insufficient Roof Space (Quantity)",
    
    bins=7
).add_to(m)

# Tooltips with useful information
GeoJson(
    gdf_map,
    style_function=lambda x: {
        "fillColor": "transparent",
        "color": "transparent",
        "weight": 0
    },
    tooltip=GeoJsonTooltip(
        fields=["municipality", "Insufficient", "Total_Buildings", "% Insufficient"],
        aliases=[
            "Municipio:",
            "Buildings with Insufficient Roof:",
            "Total Buildings:",
            "% Insufficient:"
        ],
        style=("background-color: white; color: #333; "
               "font-family: Arial; font-size: 12px; padding: 8px;")
    )
).add_to(m)

LayerControl().add_to(m)

# Save the map
m.save("PR_Roof_Space_Choropleth_Quantity.html")
print("Map saved as → PR_Roof_Space_Choropleth_Quantity.html")
m

Map saved as → PR_Roof_Space_Choropleth_Quantity.html


In [22]:
import folium
from folium import Choropleth, GeoJson, GeoJsonTooltip, LayerControl
import os

# Create outputs folder if it doesn't exist
os.makedirs("outputs", exist_ok=True)

# Base map
m = folium.Map(
    location=[18.22, -66.48],
    zoom_start=9,
    tiles="CartoDB positron"
)

# Choropleth by QUANTITY (absolute numbers)
Choropleth(
    geo_data=gdf_map.__geo_interface__,
    data=gdf_map,
    columns=["municipality", "Insufficient"],
    key_on="feature.properties.municipality",
    
    # Green → Yellow → Red
    fill_color="RdYlGn_r",
    fill_opacity=0.78,
    line_opacity=0.6,
    line_color="#333333",
    line_weight=0.7,
    
    legend_name="Number of Buildings with Insufficient Roof Space",
    name="Insufficient Roof Space (Quantity)",
    
    bins=7
).add_to(m)

# Tooltips
GeoJson(
    gdf_map,
    style_function=lambda x: {
        "fillColor": "transparent",
        "color": "transparent",
        "weight": 0
    },
    tooltip=GeoJsonTooltip(
        fields=["municipality", "Insufficient", "Total_Buildings", "% Insufficient"],
        aliases=[
            "Municipio:",
            "Buildings with Insufficient Roof:",
            "Total Buildings:",
            "% Insufficient:"
        ],
        style=(
            "background-color: white; color: #333; "
            "font-family: Arial; font-size: 12px; padding: 8px;"
        )
    )
).add_to(m)

LayerControl().add_to(m)

# Save the map
m.save("outputs/PR_Roof_Space_Choropleth_Quantity.html")
print("Map saved as → outputs/PR_Roof_Space_Choropleth_Quantity.html")

# Display the map
m

Map saved as → outputs/PR_Roof_Space_Choropleth_Quantity.html


In [23]:
# ============================================================
# PREPARE PUERTO RICO ROOFTOP DATA FOR TABLEAU
# ============================================================
import pandas as pd
import numpy as np

# ----------------------------------------------------------
# 1. Start from your municipality summary
# ----------------------------------------------------------
df = df_summary.copy()

# ----------------------------------------------------------
# 2. Clean column names (Tableau-friendly: no spaces issues OK,
#    but clear names help)
# ----------------------------------------------------------
df = df.rename(columns={
    "municipality": "Municipality",
    "num_planes": "Num_Planes",
    "num_buildings": "Num_Buildings",
    "total_flat_m2": "Total_Flat_m2",
    "total_slope_m2": "Total_Slope_m2",
    "avg_plane_flat_m2": "Avg_Plane_Flat_m2",
    "avg_plane_slope_m2": "Avg_Plane_Slope_m2",
    "total_annual_kwh": "Total_Annual_kWh",
    "total_flat_ft2": "Total_Flat_ft2",
    "total_slope_ft2": "Total_Slope_ft2",
})

# ----------------------------------------------------------
# 3. Ensure correct data types
# ----------------------------------------------------------
df["Municipality"] = df["Municipality"].astype(str).str.strip()

numeric_cols = [
    "Num_Planes", "Num_Buildings",
    "Total_Flat_m2", "Total_Slope_m2",
    "Avg_Plane_Flat_m2", "Avg_Plane_Slope_m2",
    "Total_Annual_kWh",
    "Total_Flat_ft2", "Total_Slope_ft2",
]
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# ----------------------------------------------------------
# 4. Useful calculated fields for Tableau
# ----------------------------------------------------------
df["Total_Roof_m2"] = df["Total_Flat_m2"] + df["Total_Slope_m2"]
df["Total_Roof_ft2"] = df["Total_Flat_ft2"] + df["Total_Slope_ft2"]

# Share of flat vs slope roof area
df["Pct_Flat"] = (df["Total_Flat_m2"] / df["Total_Roof_m2"] * 100).round(1)
df["Pct_Slope"] = (df["Total_Slope_m2"] / df["Total_Roof_m2"] * 100).round(1)

# Intensity metrics
df["kWh_per_Building"] = (df["Total_Annual_kWh"] / df["Num_Buildings"]).round(0)
df["kWh_per_Plane"] = (df["Total_Annual_kWh"] / df["Num_Planes"]).round(0)
df["Planes_per_Building"] = (df["Num_Planes"] / df["Num_Buildings"]).round(2)

# Capacity proxy (if 1 kW ~ rough scale from annual_kwh; optional)
# Using ~1,400 kWh/kW-year as a simple PR solar estimate
df["Est_Capacity_kW"] = (df["Total_Annual_kWh"] / 1400).round(1)

# Rank for dashboards
df["Rank_by_kWh"] = df["Total_Annual_kWh"].rank(ascending=False, method="dense").astype(int)
df["Rank_by_Buildings"] = df["Num_Buildings"].rank(ascending=False, method="dense").astype(int)

# ----------------------------------------------------------
# 5. Optional: region grouping (helps Tableau filters)
# ----------------------------------------------------------
metro = {"San Juan", "Bayamón", "Carolina", "Guaynabo", "Cataño",
         "Trujillo Alto", "Toa Baja", "Toa Alta", "Caguas", "Gurabo"}
north = {"Arecibo", "Barceloneta", "Manatí", "Vega Baja", "Vega Alta",
         "Dorado", "Hatillo", "Camuy", "Quebradillas", "Isabela"}
south = {"Ponce", "Juana Díaz", "Yauco", "Guayanilla", "Peñuelas",
         "Santa Isabel", "Salinas", "Guayama", "Arroyo", "Patillas"}
west = {"Mayagüez", "Aguadilla", "Aguada", "Rincón", "Añasco",
        "Hormigueros", "Cabo Rojo", "San Germán", "Lajas", "Sabana Grande"}
east = {"Humacao", "Yabucoa", "Maunabo", "Naguabo", "Ceiba",
        "Fajardo", "Luquillo", "Río Grande", "Canóvanas", "Loíza",
        "Juncos", "Las Piedras", "San Lorenzo"}
islands = {"Vieques", "Culebra"}

def assign_region(m):
    if m in metro: return "Metro"
    if m in north: return "North"
    if m in south: return "South"
    if m in west: return "West"
    if m in east: return "East"
    if m in islands: return "Islands"
    return "Central"

df["Region"] = df["Municipality"].apply(assign_region)

# ----------------------------------------------------------
# 6. Column order (nice for Tableau Data Source pane)
# ----------------------------------------------------------
col_order = [
    "Municipality", "Region",
    "Num_Buildings", "Num_Planes", "Planes_per_Building",
    "Total_Flat_m2", "Total_Slope_m2", "Total_Roof_m2",
    "Total_Flat_ft2", "Total_Slope_ft2", "Total_Roof_ft2",
    "Pct_Flat", "Pct_Slope",
    "Avg_Plane_Flat_m2", "Avg_Plane_Slope_m2",
    "Total_Annual_kWh", "kWh_per_Building", "kWh_per_Plane",
    "Est_Capacity_kW",
    "Rank_by_kWh", "Rank_by_Buildings",
]
df = df[[c for c in col_order if c in df.columns]]

# ----------------------------------------------------------
# 7. Quick quality check
# ----------------------------------------------------------
print("Rows:", len(df))
print("Municipalities:", df["Municipality"].nunique())
print("Nulls:\n", df.isna().sum())
print("\nPreview:")
print(df.head())

# ----------------------------------------------------------
# 8. Export for Tableau
#    Prefer CSV for large data; Excel is fine for 78 rows
# ----------------------------------------------------------
df.to_csv("Tableau_PR_Rooftop_Municipalities.csv", index=False, encoding="utf-8-sig")
df.to_excel("Tableau_PR_Rooftop_Municipalities.xlsx", index=False)

print("\nSaved:")
print("  Tableau_PR_Rooftop_Municipalities.csv")
print("  Tableau_PR_Rooftop_Municipalities.xlsx")

Rows: 78
Municipalities: 78
Nulls:
 Municipality           0
Region                 0
Num_Buildings          0
Num_Planes             0
Planes_per_Building    0
Total_Flat_m2          0
Total_Slope_m2         0
Total_Roof_m2          0
Total_Flat_ft2         0
Total_Slope_ft2        0
Total_Roof_ft2         0
Pct_Flat               0
Pct_Slope              0
Avg_Plane_Flat_m2      0
Avg_Plane_Slope_m2     0
Total_Annual_kWh       0
kWh_per_Building       0
kWh_per_Plane          0
Est_Capacity_kW        0
Rank_by_kWh            0
Rank_by_Buildings      0
dtype: int64

Preview:
   Municipality   Region  Num_Buildings  Num_Planes  Planes_per_Building  \
0      Adjuntas  Central           6327       33602                 5.31   
1        Aguada     West          18089      107466                 5.94   
2     Aguadilla     West          23698      147710                 6.23   
3  Aguas Buenas  Central          10858       63179                 5.82   
4      Aibonito  Central          12

In [24]:
def export_municipality_for_tableau(name: str):
    df_m = load_municipality(name)  # your existing function
    # keep useful columns only
    keep = [c for c in [
        "bldg_fid", "azimuth", "tilt", "flat_m2", "slope_m2",
        "slope_degrees", "pct_shading", "kw", "annual_kwh", "county"
    ] if c in df_m.columns]
    out = df_m[keep].copy()
    out.to_csv(f"Tableau_PR_Planes_{name.replace(' ', '_')}.csv", index=False, encoding="utf-8-sig")
    print(f"Saved Tableau_PR_Planes_{name.replace(' ', '_')}.csv  ({len(out):,} rows)")
    return out

# Example:
# export_municipality_for_tableau("Caguas")

In [25]:
# ============================================================
# DATA FOR TABLEAU MAPS  (same logic as your Folium choropleths)
# ============================================================
import duckdb
import pandas as pd
import numpy as np
import unicodedata

# ----------------------------------------------------------
# CONFIG (same as your notebook)
# ----------------------------------------------------------
MIN_AREA_FT2 = 130          # minimum suitable roof (ft²)
USE_SLOPE_AREA = True       # True = slope_m2, False = flat_m2
M2_TO_FT2 = 10.7639

# ----------------------------------------------------------
# 1. Building-level classification (same query you used)
# ----------------------------------------------------------
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")
con.execute("SET s3_access_key_id='';")
con.execute("SET s3_secret_access_key='';")

area_col = "slope_m2" if USE_SLOPE_AREA else "flat_m2"

query = f"""
SELECT 
    county AS municipality,
    bldg_fid,
    COUNT(*) AS num_planes,
    ROUND(SUM(flat_m2), 2) AS total_flat_m2,
    ROUND(SUM(slope_m2), 2) AS total_slope_m2,
    ROUND(SUM(flat_m2) * {M2_TO_FT2}, 1) AS total_flat_ft2,
    ROUND(SUM(slope_m2) * {M2_TO_FT2}, 1) AS total_slope_ft2,
    ROUND(SUM(annual_kwh), 0) AS total_annual_kwh,
    CASE 
        WHEN SUM({area_col}) * {M2_TO_FT2} >= {MIN_AREA_FT2} 
        THEN 'Suitable' 
        ELSE 'Insufficient' 
    END AS roof_status
FROM read_parquet(
    's3://oedi-data-lake/pv-rooftop-pr/developable-planes/county=*/**/*.parquet',
    hive_partitioning = 1
)
GROUP BY county, bldg_fid
"""

df_buildings = con.execute(query).df()
print(f"Buildings: {len(df_buildings):,}")

# ----------------------------------------------------------
# 2. Municipality summary (THIS is what the Folium map used)
# ----------------------------------------------------------
summary = (
    df_buildings
    .groupby(["municipality", "roof_status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# ensure both status columns exist
for col in ["Suitable", "Insufficient"]:
    if col not in summary.columns:
        summary[col] = 0

summary["Total_Buildings"] = summary["Suitable"] + summary["Insufficient"]
summary["% Suitable"] = (summary["Suitable"] / summary["Total_Buildings"] * 100).round(1)
summary["% Insufficient"] = (summary["Insufficient"] / summary["Total_Buildings"] * 100).round(1)
summary["Min_Area_ft2"] = MIN_AREA_FT2

# ----------------------------------------------------------
# 3. Clean municipality names (same cleaning as for the map join)
# ----------------------------------------------------------
def clean_muni_name(name):
    if pd.isna(name):
        return name
    name = unicodedata.normalize("NFKD", str(name)).encode("ASCII", "ignore").decode("utf-8")
    name = name.replace(" Municipio", "").replace("Municipio", "").strip().title()
    return name

summary["municipality"] = summary["municipality"].apply(clean_muni_name)

# optional accent-free fixes (same idea as your name_fix)
name_fix = {
    "Anasco": "Anasco",
    "Rincon": "Rincon",
    "Guanica": "Guanica",
    "San Sebastian": "San Sebastian",
    "Loiza": "Loiza",
    "Manati": "Manati",
    "Mayaguez": "Mayaguez",
    "Bayamon": "Bayamon",
    "Catano": "Catano",
    "Juana Diaz": "Juana Diaz",
    "Penuelas": "Penuelas",
}
summary["municipality"] = summary["municipality"].replace(name_fix)

# ----------------------------------------------------------
# 4. Join solar capacity from df_summary (optional but useful)
# ----------------------------------------------------------
# If you still have df_summary from earlier cells:
try:
    solar = df_summary.copy()
    solar["municipality"] = solar["municipality"].apply(clean_muni_name)
    solar = solar.rename(columns={
        "num_planes": "Num_Planes_Total",
        "num_buildings": "Num_Buildings_Solar",
        "total_annual_kwh": "Total_Annual_kWh",
        "total_flat_m2": "Total_Flat_m2",
        "total_slope_m2": "Total_Slope_m2",
    })
    keep_solar = ["municipality", "Num_Planes_Total", "Total_Annual_kWh",
                  "Total_Flat_m2", "Total_Slope_m2"]
    keep_solar = [c for c in keep_solar if c in solar.columns]
    summary = summary.merge(solar[keep_solar], on="municipality", how="left")
except NameError:
    print("df_summary not in memory — skipping solar join")

# ----------------------------------------------------------
# 5. Tableau-friendly column names + order
# ----------------------------------------------------------
summary = summary.rename(columns={
    "municipality": "Municipality",
    "Suitable": "Suitable_Buildings",
    "Insufficient": "Insufficient_Buildings",
    "Total_Buildings": "Total_Buildings",
    "% Suitable": "Pct_Suitable",
    "% Insufficient": "Pct_Insufficient",
})

# ranks for maps/legends
summary["Rank_Pct_Insufficient"] = summary["Pct_Insufficient"].rank(ascending=False, method="dense").astype(int)
summary["Rank_Insufficient_Count"] = summary["Insufficient_Buildings"].rank(ascending=False, method="dense").astype(int)

cols = [
    "Municipality",
    "Suitable_Buildings", "Insufficient_Buildings", "Total_Buildings",
    "Pct_Suitable", "Pct_Insufficient", "Min_Area_ft2",
    "Rank_Pct_Insufficient", "Rank_Insufficient_Count",
    "Num_Planes_Total", "Total_Annual_kWh", "Total_Flat_m2", "Total_Slope_m2",
]
summary = summary[[c for c in cols if c in summary.columns]]
summary = summary.sort_values("Pct_Insufficient", ascending=False)

print(summary.head(10))
print(f"\nMunicipalities: {len(summary)}")

# ----------------------------------------------------------
# 6. EXPORT FOR TABLEAU
# ----------------------------------------------------------
# A) Municipality choropleth data (same metrics as Folium)
summary.to_csv("Tableau_PR_Map_Roof_Sufficiency.csv", index=False, encoding="utf-8-sig")
summary.to_excel("Tableau_PR_Map_Roof_Sufficiency.xlsx", index=False)

# B) Building-level detail (for drill-down, filters, scatter)
cols_bldg = [
    "municipality", "bldg_fid", "num_planes",
    "total_flat_ft2", "total_slope_ft2", "total_annual_kwh", "roof_status"
]
df_buildings_out = df_buildings[cols_bldg].copy()
df_buildings_out = df_buildings_out.rename(columns={
    "municipality": "Municipality",
    "bldg_fid": "Building_ID",
    "num_planes": "Num_Planes",
    "total_flat_ft2": "Flat_ft2",
    "total_slope_ft2": "Slope_ft2",
    "total_annual_kwh": "Annual_kWh",
    "roof_status": "Roof_Status",
})
df_buildings_out["Municipality"] = df_buildings_out["Municipality"].apply(clean_muni_name)
df_buildings_out.to_csv("Tableau_PR_Buildings_Roof_Status.csv", index=False, encoding="utf-8-sig")

print("\nSaved for Tableau:")
print("  Tableau_PR_Map_Roof_Sufficiency.csv / .xlsx   ← use this for the map")
print("  Tableau_PR_Buildings_Roof_Status.csv          ← building-level detail")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Buildings: 1,343,874
   Municipality  Suitable_Buildings  Insufficient_Buildings  Total_Buildings  \
41        Lares               14360                    1142            15502   
47      Maricao                3135                     246             3381   
42   Las Marias                4011                     314             4325   
71       Utuado               14857                    1128            15985   
56     Penuelas                8781                     660             9441   
54     Orocovis               11291                     835            12126   
7        Anasco               12786                     934            13720   
0      Adjuntas                5901                     426             6327   
32      Guanica                8011                     575             8586   
63   San German               13213                     947            14160   

    Pct_Suitable  Pct_Insufficient  Min_Area_ft2  Rank_Pct_Insufficient  \
41          92.6       

In [26]:
# Optional: export the same polygons Folium used
gdf_map_export = gdf_map[[
    "municipality", "Suitable", "Insufficient", "Total_Buildings",
    "% Suitable", "% Insufficient", "geometry"
]].copy() if "geometry" in gdf_map.columns else None

if gdf_map_export is not None:
    gdf_map_export.to_file("Tableau_PR_Municipios_Roof.geojson", driver="GeoJSON")
    print("Saved Tableau_PR_Municipios_Roof.geojson")

Saved Tableau_PR_Municipios_Roof.geojson
